# Imports

In [ ]:
from huggingface_hub import login

hf_token = "token"
login(hf_token)

/data/storage1/SpanishDialectDiscrimination/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch
import pandas as pd
import numpy as np
import random

# Extracting and randomizing adjectives

In [3]:
adjs_df = pd.read_csv("Data/adjective_dataset_v3.csv")
adjs_df.head()

,Inteligent_ES,Inteligent_EN,Uninteligent_ES,Uninteligent_EN,SelfConfident_ES,SelfConfident_EN,Insecure_ES,Insecure_EN,Trustworthy_ES,Trustworthy_EN,...,Unfriendly_ES,Unfriendly_EN,Kind_ES,Kind_EN,Unkind_ES,Unkind_EN,Outgoing_ES,Outgoing_EN,Antisocial_ES,Antisocial_EN
0,inteligente,intelligent,tonto,unintelligent,seguro,self-confident,inseguro,insecure,fiable,trustworthy,...,hostil,unfriendly,amable,kind,rudo,unkind,extrovertido,outgoing,introvertido,antisocial
1,listo,smart,torpe,dumb,decidido,confident,inestable,worried,fiel,reliable,...,contrario,icy,atento,compassionate,brusco,rude,extravertido,social,retraído,detached
2,capaz,clever,incapaz,stupid,aplomado,optimistic,indeciso,nervous,leal,responsible,...,enemigo,cold,benévolo,benevolent,ordinario,unpleasant,comunicativo,extroverted,tímido,distant
3,ingenioso,quick,inepto,slow,resuelto,assured,vacilante,anxious,confiable,safe,...,adverso,frigid,cariñoso,thoughtful,áspero,unfavorable,sociable,extraverted,insociable,unsociable
4,sesudo,brilliant,incompetente,simple,confiado,self-assured,incierto,upset,fidedigno,true,...,rival,brittle,afectuoso,sympathetic,grosero,inconsiderate,abierto,gregarious,reservado,asocial


In [4]:
adjs = np.concatenate((adjs_df['Inteligent_ES'].values, adjs_df['Uninteligent_ES'].values))

np.random.shuffle(adjs)

adjs

array(['suspenso', 'sagaz', 'apto', 'despabilado', 'torpe', 'avispado',
       'espabilado', 'agudo', 'lento', 'inexperto', 'inteligente',
       'astuto', 'vivo', 'incapaz', 'idóneo', 'incompetente', 'tonto',
       'ingenioso', 'soso', 'calificado', 'cándido', 'inapto', 'bobo',
       'capaz', 'ingenuo', 'obtuso', 'desacreditado', 'despejado',
       'despierto', 'irreflexivo', 'alocado', 'perspicaz', 'atontado',
       'inadecuado', 'inepto', 'inapropiado', 'sesudo', 'competente',
       'listo', 'suficiente'], dtype=object)

In [5]:
adj_str = ', '.join(map(str,adjs))
adj_str

'suspenso, sagaz, apto, despabilado, torpe, avispado, espabilado, agudo, lento, inexperto, inteligente, astuto, vivo, incapaz, idóneo, incompetente, tonto, ingenioso, soso, calificado, cándido, inapto, bobo, capaz, ingenuo, obtuso, desacreditado, despejado, despierto, irreflexivo, alocado, perspicaz, atontado, inadecuado, inepto, inapropiado, sesudo, competente, listo, suficiente'

In [26]:
en_adjs = np.concatenate((adjs_df['Inteligent_EN'].values, adjs_df['Uninteligent_EN'].values))

np.random.shuffle(en_adjs)

en_adj_str = ', '.join(map(str,en_adjs))
en_adj_str

'exceptional, sensible, mindless, alert, doltish, literate, obtuse, fatuous, resourceful, clever, thick, brainless, fast, cunning, idiotic, nimble, wise, quick, unintelligent, witless, skilled, bright, irrational, dull, smart, keen, ignorant, slow, oafish, dumb, intelligent, sharp, foolish, educated, vacuous, stupid, brilliant, brainy, simple, dense'

# Extract Sentences

In [6]:
sen_df = pd.read_csv("Data/Working_Sentence_Dataset.csv")

sen_df['Sentence_ID'] = range(len(sen_df))
sen_df['Sentence_ID'] += 1 

sen_df.head()

,Sentence_ID,Text ID,Line,Speaker,Raw,PS,PS_Check,PS_Note,MS,MS_Check,MS_Note,English,Type
0,1,ALCA_H11_037,10,B,¿qué has hecho hoy? / cuéntame <silencio/> cur...,¿Qué has hecho hoy? Cuéntame - Currar y pasar ...,NaN,NaN,¿Qué hiciste hoy? Cuéntame - Chambear y pasar ...,NaN,NaN,What did you do today? Tell me - I worked and ...,B
1,2,ALCA_H11_037,11,I,he cambiado / porque la otra vez / la niña est...,"He cambiado, porque la niña estaba en el coleg...",NaN,NaN,"Cambié, porque la niña estaba en la escuela, p...",X,"Cambié, porque la niña estaba en la escuela, p...","I changed, because the girl was in school, but...",B
2,3,ALCA_H11_037,12,I,hoy me ha tocado / irme con el coche a un pueb...,Hoy me ha tocado irme con el coche a un pueblo...,NaN,NaN,Hoy me tocó irme con el carro a un pueblo a re...,NaN,NaN,Today I had to go to a town by car to deliver ...,Both
3,4,ALCA_H11_037,14,I,normalmente si estoy solo si tengo jaleo pues ...,"Normalmente, si estoy solo o si tengo jaleo, p...",NaN,NaN,"Normalmente, si estoy solo o si hay alboroto, ...",NaN,NaN,"Usually, if I'm alone or if there's a commotio...",L
4,5,ALCA_H11_037,16,I,le vamos a coger el gustillo,Le vamos a coger el gustillo,NaN,NaN,Le vamos a agarrar el gustito,NaN,NaN,We're going to take a liking to it,L


In [7]:
sen_only = sen_df[['Sentence_ID','PS','MS']]
sen_only.head()

,Sentence_ID,PS,MS
0,1,¿Qué has hecho hoy? Cuéntame - Currar y pasar ...,¿Qué hiciste hoy? Cuéntame - Chambear y pasar ...
1,2,"He cambiado, porque la niña estaba en el coleg...","Cambié, porque la niña estaba en la escuela, p..."
2,3,Hoy me ha tocado irme con el coche a un pueblo...,Hoy me tocó irme con el carro a un pueblo a re...
3,4,"Normalmente, si estoy solo o si tengo jaleo, p...","Normalmente, si estoy solo o si hay alboroto, ..."
4,5,Le vamos a coger el gustillo,Le vamos a agarrar el gustito


In [8]:
id = 10
current_row = sen_only.loc[sen_only['Sentence_ID'] == id]
ps_sen = current_row['PS'][id-1]
ms_sen = current_row['MS'][id-1]

print(id)
print("PS: ", ps_sen)
print("MS: ", ms_sen)


10
PS:  No hemos tenido jaleos con los del pueblo
MS:  No hemos tenido bronca con los del pueblo


# Run Model

In [9]:
model_id = "google/gemma-3-12b-it"

In [10]:
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 5/5 [00:18<00:00,  3.74s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


# Query Consecutively

**Spanish**

In [28]:
sp_result_data = []

sp_prompt = "Aqui tienes una lista de adjetivos. Por cada adjetivo, escoge un escritor - A o B - y escribelo junto al adjetivo sin explicacin adicional. Pon una coma despues de cada escritor. Los adjetivos son: "

In [29]:
for i in range(11,51):

    run_data = []
    run_data.append(i)

    # randomise adjective list
    np.random.shuffle(adjs)
    adj_str = ', '.join(map(str,adjs))

    # get current sentence
    current_row = sen_only.loc[sen_only['Sentence_ID'] == i]
    ps_sen = current_row['PS'][i-1]
    ms_sen = current_row['MS'][i-1]

    # write full prompt, alternatig order of ps and ms sentences
    if i%2 == 1:
        full_prompt = sp_prompt + adj_str + ". " + "A: " + ps_sen + " B: " + ms_sen
        run_data.append("PS")
        run_data.append("MS")
    else: 
        full_prompt = sp_prompt + adj_str + ". " + "A: " + ms_sen + " B: " + ps_sen
        run_data.append("MS")
        run_data.append("PS")
    run_data.append(full_prompt)

    # query model

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "Eres un asistente."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": full_prompt}
            ]
        }
    ]





    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=500, do_sample=False)
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    run_data.append(decoded)

    # append results of run
    sp_result_data.append(run_data)


In [30]:
sp_results_df = pd.DataFrame(sp_result_data, columns=["sen_id", "A", "B", "prompt", "response"])

sp_results_df.head()

,sen_id,A,B,prompt,response
0,11,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Alocado, B. Irreflexivo, B. Ingenuo, A. Sesudo..."
1,12,MS,PS,Aqui tienes una lista de adjetivos. Por cada a...,"Calificado, A\nAtonitado, B\nIncapaz, B\nIrref..."
2,13,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Ingenioso, B\nDespierto, B\nInexperto, A\nInap..."
3,14,MS,PS,Aqui tienes una lista de adjetivos. Por cada a...,"Torpe, B, Sagaz, A, Inexperto, B, Despejado, A..."
4,15,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Desacreditado, B\nInexperto, A\nTonto, B\nTorp..."


**English**

In [31]:
en_result_data = []

en_prompt = "Here is a list of adjectives. For each adjective, chose a writer - A o B - and write it next to the adjective without further explanation. Put a comma fter each writer. The adjectives are: "

In [32]:
for i in range(11,51):

    run_data = []
    run_data.append(i)

    # randomise adjective list
    np.random.shuffle(en_adjs)
    en_adj_str = ', '.join(map(str,en_adjs))

    # get current sentence
    current_row = sen_only.loc[sen_only['Sentence_ID'] == i]
    ps_sen = current_row['PS'][i-1]
    ms_sen = current_row['MS'][i-1]

    # write full prompt, alternatig order of ps and ms sentences
    if i%2 == 1:
        full_prompt = en_prompt + en_adj_str + ". " + "A: " + ps_sen + " B: " + ms_sen
        run_data.append("PS")
        run_data.append("MS")
    else: 
        full_prompt = en_prompt + en_adj_str + ". " + "A: " + ms_sen + " B: " + ps_sen
        run_data.append("MS")
        run_data.append("PS")
    run_data.append(full_prompt)

    # query model

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "Eres un asistente."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": full_prompt}
            ]
        }
    ]





    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=500, do_sample=False)
        generation = generation[0][input_len:]

    decoded = processor.decode(generation, skip_special_tokens=True)
    run_data.append(decoded)

    # append results of run
    en_result_data.append(run_data)


In [33]:
en_results_df = pd.DataFrame(en_result_data, columns=["sen_id", "A", "B", "prompt", "response"])

en_results_df.head()

,sen_id,A,B,prompt,response
0,11,PS,MS,Here is a list of adjectives. For each adjecti...,"nimble, A\nbright, B\nfoolish, A\nkeen, B\nwis..."
1,12,MS,PS,Here is a list of adjectives. For each adjecti...,"brilliant, A\nunintelligent, B\ndull, B\nskill..."
2,13,PS,MS,Here is a list of adjectives. For each adjecti...,Here's the list of adjectives paired with eith...
3,14,MS,PS,Here is a list of adjectives. For each adjecti...,Here's the list of adjectives paired with eith...
4,15,PS,MS,Here is a list of adjectives. For each adjecti...,Here's the list of adjectives paired with eith...


# Testing

In [17]:
test = results_df["response"][1]

In [22]:
test2 = test.replace(", B", " B,")
test2 = test2.replace(", A", " A,")
test2 = test2.replace("\n","")
test2

test2.split(',')

['incapaz B',
 'calificado A',
 'inepto B',
 'apto A',
 'cándido B',
 'soso A',
 'astuto B',
 'listo A',
 'suspenso B',
 'ingenioso A',
 'vivo B',
 'atontado A',
 'desacreditado B',
 'inapropiado A',
 'incompetente B',
 'tonto A',
 'suficiente B',
 'inexperto A',
 'competente B',
 'despejado A',
 'despabilado B',
 'despierto A',
 'capaz B',
 'irreflexivo A',
 'avispado B',
 'sagaz A',
 'sesudo B',
 'espabilado A',
 'lento B',
 'alocado A',
 'inadecuado B',
 'agudo A',
 'idóneo B',
 'inapto A',
 'obtuso B',
 'inteligente A',
 'ingenuo B',
 'perspicaz A',
 'torpe B',
 'bobo A',
 '']

In [40]:

sp_results_df.to_csv('results_gemma__AssocTask_intelligent_sp.csv', index=False)

en_results_df.to_csv('results_gemma__AssocTask_intelligent_en.csv', index=False)

In [42]:
t = pd.read_csv('results_gemma__AssocTask_intelligent_sp.csv',)

t.head()

,sen_id,A,B,prompt,response
0,11,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Alocado, B. Irreflexivo, B. Ingenuo, A. Sesudo..."
1,12,MS,PS,Aqui tienes una lista de adjetivos. Por cada a...,"Calificado, A\nAtonitado, B\nIncapaz, B\nIrref..."
2,13,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Ingenioso, B\nDespierto, B\nInexperto, A\nInap..."
3,14,MS,PS,Aqui tienes una lista de adjetivos. Por cada a...,"Torpe, B, Sagaz, A, Inexperto, B, Despejado, A..."
4,15,PS,MS,Aqui tienes una lista de adjetivos. Por cada a...,"Desacreditado, B\nInexperto, A\nTonto, B\nTorp..."


# Run Model

Loading checkpoint shards: 100%|██████████| 5/5 [00:18<00:00,  3.77s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


**Spanish**

In [27]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "Eres un asistente."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Aqui tienes una lista de adjetivos. Por cada adjetivo, escoge un escritor - A o B - y escribelo junto al adjetivo sin explicacin adicional. Pon una coma despues de cada escritor. Los adjetivos son: \
               atontado, suficiente, cándido, despierto, competente, incapaz, inepto, sesudo, alocado, incompetente, bobo, soso, capaz, avispado, desacreditado, inapto, lento, agudo, ingenuo, calificado, idóneo, suspenso, despabilado, espabilado, sagaz, vivo, listo, irreflexivo, inteligente, tonto, astuto, inexperto, apto, inapropiado, torpe, obtuso, despejado, perspicaz, inadecuado, ingenioso. \
                A: No hemos tenido bronca con los del pueblo \
                B: No hemos tenido jaleos con los del pueblo"}
        ]
    }
]





inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

Atontado, B
Suficiente, A
Cándido, B
Despierto, A
Competente, A
Incapaz, B
Inepto, B
Sesudo, B
Alocado, B
Incompetente, B
Bobo, B
Soso, B
Capaz, A
Avispado, A
Desacreditado, B
Inapto, B
Lento, B
Agudo, A
Ingenuo, B
Calificado, A
Idóneo, A
Suspenso, B
Despabilado, A
Espabilado, A
Sagaz, A
Vivo, A
Listo, A
Irreflexivo, B
Inteligente, A
Tonto, B
Astuto, A
Inexperto, B
Apto, A
Inapropiado, B
Torpe, B
Obtuso, B
Despejado, A
Perspicaz, A
Inadecuado, B
Ingenioso, A


**English**

In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "Eres un asistente."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Here is a list of adjectives. Fore each adjective, pick a writer - A o B - and write it next to the adjective without additional explanatgion. The adjectives are: \
                cándido,competente,torpe,inteligente,idóneo,capaz,astuto,sagaz,irreflexivo,inapropiado,ingenioso,incompetente,alocado,obtuso,lento,despierto,suspenso,avispado,inepto,bobo,inadecuado,suficiente,despejado,inexperto,incapaz,apto,inapto,espabilado,agudo,perspicaz,listo,tonto,soso,despabilado,vivo,atontado,desacreditado,calificado,sesudo,ingenuo \
                A:  ¿Qué has hecho hoy? Cuéntame - Currar y pasar mucho calor \
                B:  ¿Qué hiciste hoy? Cuéntame - Chambear y pasar mucho calor "}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

In [7]:
model_reply = decoded.strip()

In [8]:
messages.append({
    "role":"assistant",
    "content": [{"type": "text", "text": model_reply}]
})

messages.append({
    "role":"user",
    "content": [{"type": "text", "text": "Explica tu respuesta"}]
})

In [9]:
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

Mi selección de escritores A y B se basa en una percepción general de sus estilos y temas recurrentes en sus obras. No es una regla rígida, y podría haber excepciones, pero esta es la lógica detrás de mis elecciones:

**Escritor A (Asumo que es un escritor con una inclinación hacia la inteligencia, la agudeza mental y la perspicacia):**

*   **Adjetivos asociados:** Los adjetivos que asigné al escritor A tienden a describir personas o situaciones que demuestran inteligencia, rapidez mental, habilidad y perspicacia. Esto incluye palabras como "smart," "brainy," "alert," "intelligent," "brilliant," "exceptional," "wise," "quick," "educated," "sensible," "sharp," "bright," "nimble," "keen," y "literate."  Un escritor que se centra en personajes inteligentes, que resuelven problemas con astucia, o que exploran ideas complejas, encajaría mejor con estos adjetivos.

**Escritor B (Asumo que es un escritor que se enfoca en personajes con falta de inteligencia, torpeza, o comportamiento irracio

In [ ]:
changer = TokenizerChanger(tokenizer)